###Referenz ⇒ Canonical Form 
- pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
- bei SAbPred: SCALOP in Submission form die Datei hochladen 
- results: für jedes CDR (H1, H2, L1, L2, L3) erkennt er CDR Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
- (L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
- muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
- dann der vergleich mit V-measure

'''# SCALOP Installation und Setup Anleitung
# 1. Projekt klonen (wenn noch nicht gemacht)
git clone https://github.com/oxpig/SCALOP.git

# 2. Conda-Umgebung erstellen
conda create -n scalop-env python=3.8 -y

# 3. Conda initialisieren (falls noch nie gemacht)
conda init
# dann Terminal neu starten
exit
# dann Terminal wieder öffnen

# 4. Umgebung aktivieren
conda activate scalop-env

# 5. Abhängigkeiten installieren
conda install -c bioconda numpy biopython -y
conda install -c bioconda hmmer -y   # funktioniert nur in WSL-Umgebung, nicht in Windows direkt! Wird für assign benötigt

# 6. SCALOP lokal installieren
pip install ./SCALOP

# 7. Kernel registrieren (für Jupyter-Notebook))
pip install ipykernel
python -m ipykernel install --user --name scalop-env --display-name "Python (scalop-env)"
# dann Jupyter Notebook öffnen und den Kernel "Python (scalop-env)" auswählen. Jetzt kann man SCALOP in Jupyter Notebooks verwenden.'''

In [2]:
from scalop.predict import assign
input='VKLLEQSGAEVKKPGASVKVSCKASGYSFTSYGLHWVRQAPGQRLEWMGWISAGTGNTKYSQKFRGRVTFTRDTSATTAYMGLSSLRPEDTAVYYCARDPYGGGKSEFDYWGQGTLVTVSS/ELVMTQSPSSLSASVGDRVNIACRASQGISSALAWYQQKPGKAPRLLIYDASNLESGVPSRFSGSGSGTDFTLTISSLQPEDFAIYYCQQFNSYPLTFGGGTKVEIKRTV'
assign(input)

import csv


In [8]:
from pprint import pprint
from scalop.predict import assign

input_seq = 'VKLLEQSGAEVKKPGASVKVSCKASGYSFTSYGLHWVRQAPGQRLEWMGWISAGTGTKYSQKFRGRVTFTFRDTSATTAYMGLSSLRPEDTAVVYCARDPYGGGKSEFDYWQGQTLVTVSS/ELVMTQSPSSLSASVGDRVNIACRASQGISSALAWYQQKPGKAPRLLIYDASNLESGVPSRFGSGSGTDFTLTISSLQPEDFAIYYCQQFNSYPLTFGGGTKVEIKRTV'

result = assign(input_seq, scheme="chothia", definition="chothia")
pprint(result)

[{'input': ('input_1',
            'VKLLEQSGAEVKKPGASVKVSCKASGYSFTSYGLHWVRQAPGQRLEWMGWISAGTGTKYSQKFRGRVTFTFRDTSATTAYMGLSSLRPEDTAVVYCARDPYGGGKSEFDYWQGQTLVTVSS'),
  'outputs': {'H1': ['H1', 'GYSFTSY', 'H1-7-C', '6ey6_I'],
              'H2': ['H2', 'SAGTG', 'H2-5-A', '3zkm_C']},
  'seqname': 'input_1'},
 {'input': ('input_2',
            'ELVMTQSPSSLSASVGDRVNIACRASQGISSALAWYQQKPGKAPRLLIYDASNLESGVPSRFGSGSGTDFTLTISSLQPEDFAIYYCQQFNSYPLTFGGGTKVEIKRTV'),
  'outputs': {'L1': ['L1', 'RASQGISSALA', 'L1-11-A', '5gis_L'],
              'L2': ['L2', 'DASNLES', 'L2-7-A', '2g5b_A'],
              'L3': ['L3', 'QQFNSYPLT', 'L3-9,10-A', '5vpl_C']},
  'seqname': 'input_2'}]


In [11]:
import csv
from scalop.predict import assign

input_seq = 'VKLLEQSGAEVKKPGASVKVSCKASGYSFTSYGLHWVRQAPGQRLEWMGWISAGTGTKYSQKFRGRVTFTFRDTSATTAYMGLSSLRPEDTAVVYCARDPYGGGKSEFDYWQGQTLVTVSS/ELVMTQSPSSLSASVGDRVNIACRASQGISSALAWYQQKPGKAPRLLIYDASNLESGVPSRFGSGSGTDFTLTISSLQPEDFAIYYCQQFNSYPLTFGGGTKVEIKRTV'

# Aufruf
results = assign(input_seq, scheme="chothia", definition="chothia")

# CSV-Dateiname
output_file = "canonical_forms_output.csv"

# Schreibe Ergebnisse in CSV
with open(output_file, mode="w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["SeqName", "CDR", "CDR_Sequence", "Canonical_Form", "Structure"])

    for result in results:
        seqname = result['input'][0]
        outputs = result['outputs']
        for cdr, data in outputs.items():
            cdr_name = data[0]
            cdr_seq = data[1]
            canonical_form = data[2]
            structure = data[3]
            writer.writerow([seqname, cdr_name, cdr_seq, canonical_form, structure])

print(f"Alle CDRs und Canonical Forms wurden in {output_file} gespeichert.")


Alle CDRs und Canonical Forms wurden in canonical_forms_output.csv gespeichert.


In [ ]:
import sys
print(sys.executable)


c:\Users\avdh3\OneDrive\Dokumente\GitHub\group04-team04\.conda\python.exe


In [ ]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path
        

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None